<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# E-Commerce Analytics 360
## Notebook 4 — Dashboard Power BI & Storytelling

> **Prérequis** : Notebooks 1, 2 et 3 complétés. Les fichiers CSV exportés doivent être disponibles sur le poste.

| | |
|---|---|
| **Niveau** | Avancé |
| **Outils** | Power BI Desktop |
| **Durée estimée** | 6h à 8h |

> 💡 **Ce notebook est un guide de conception reproductible.** En le suivant pas à pas, tu produiras le dashboard exactement tel qu'il apparaît dans le rapport de référence (5 pages avec navigation latérale DataProjectLab E-commerce).

### Objectif business
Transformer les données SQL en un dashboard décisionnel **5 pages** permettant à M. Diallo de piloter : performance commerciale · produits · clients · funnel digital · satisfaction.

---
## 1. Sources de données (7 fichiers)

### Fichiers à importer dans Power BI

| Fichier | Type | Rôle |
|---|---|---|
| `dim_customers.csv` | Dimension | Profil clients (segment, pays, ville) |
| `dim_products.csv` | Dimension | Catalogue produits (catégorie, prix) |
| `fact_orders.csv` | Fait | Commandes (date, client, canal, statut) |
| `fact_order_items.csv` | Fait | Lignes de commande (produit, quantité, CA, marge) |
| `fact_reviews.csv` | Fait | Avis clients (rating, date, produit) |
| `fact_web_logs.csv` | Fait | Comportement web (session, page, device, source) |
| `clients_rfm_segments.csv` | Analytique | Segmentation RFM (6 segments) |

> ⚠️ **Important** : la table `fact_ecommerce_analytics` de l'étape 3 n'est **pas importée** dans ce rapport. Les analyses passent directement par les tables de faits, ce qui simplifie le modèle et évite les doublons.

### Import

### Import

1. Power BI Desktop → **Obtenir des données → Web**
2. Coller chaque URL dans la fenêtre web
3. Cliquer **Transformer les données** (pour vérifier les types)
4. Vérifier les types en Power Query avant de charger (notamment `premium`, `at_risk_dropout`, `certificat_obtenu`, `alerte_decrochage` en **Nombre entier** 0/1)
5. **Accueil → Fermer & appliquer**

#### customers

https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/corrige/outputs/dim_customers.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/corrige/outputs/dim_products.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/corrige/outputs/fact_orders.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/corrige/outputs/fact_order_items.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/corrige/outputs/fact_reviews.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/corrige/outputs/fact_web_logs.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/corrige/outputs/clients_rfm_segments.csv

---
## 2. Désactiver Auto Date/Time (obligatoire)

Avant toute manipulation du modèle :

**Fichier → Options → Chargement des données (Fichier actuel) → DÉCOCHER "Date/heure automatique pour le fichier actuel"**

Sans cette étape, Power BI crée des tables `LocalDateTable_*` parasites (5 à 10 tables selon le nombre de colonnes date) qui polluent le modèle et empêchent certaines mesures de fonctionner correctement.

---
## 3. Modèle de données — Schéma en étoile

### Architecture du modèle

```
                              Calendrier (dim temps)
                                     |
                                     v
           dim_customers --> fact_orders --> fact_order_items <-- dim_products
                  |                |
                  |                v
                  +---------> fact_reviews
                  |
                  +---------> fact_web_logs
                  |
                  +---------> clients_rfm_segments
```

### Relations à créer (10 relations)

| De | Colonne | Vers | Colonne | Cardinalité | Direction |
|---|---|---|---|---|---|
| `fact_orders` | `customer_id` | `dim_customers` | `customer_id` | N→1 | Single |
| `fact_reviews` | `customer_id` | `dim_customers` | `customer_id` | N→1 | Single |
| `fact_web_logs` | `customer_id` | `dim_customers` | `customer_id` | N→1 | Single |
| `clients_rfm_segments` | `customer_id` | `dim_customers` | `customer_id` | N→1 | Single |
| `fact_order_items` | `order_id` | `fact_orders` | `order_id` | N→1 | Single |
| `fact_order_items` | `product_id` | `dim_products` | `product_id` | N→1 | Single |
| `fact_reviews` | `order_id` | `fact_orders` | `order_id` | N→1 | Single |
| `fact_reviews` | `product_id` | `dim_products` | `product_id` | N→1 | Single |
| `fact_orders` | `order_date` | `Calendrier` | `Date` | N→1 | Single |
| `fact_web_logs` | `session_date` | `Calendrier` | `Date` | N→1 | Single |

### ⚠️ Relations à supprimer si Power BI les crée automatiquement

Vérifier dans **Affichage du modèle** qu'il n'y a **aucune relation M:M bidirectionnelle** ni **aucun chemin ambigu**. Si détecté, supprimer les relations dangling comme :

- `fact_web_logs ↔ fact_orders` (M:M bidir)
- `fact_reviews ↔ fact_order_items` (M:M bidir)

---
## 4. Table Calendrier

Modélisation → **Nouvelle table** → coller :

```dax
Calendrier =
ADDCOLUMNS(
    CALENDAR(DATE(2022,1,1), DATE(2024,12,31)),
    "Annee",        YEAR([Date]),
    "Mois_Num",     MONTH([Date]),
    "Mois_Nom",     FORMAT([Date], "MMMM", "fr-FR"),
    "Trimestre",    "T" & QUARTER([Date]),
    "Semaine",      WEEKNUM([Date]),
    "Jour_Semaine", FORMAT([Date], "dddd", "fr-FR"),
    "Est_Weekend",  IF(WEEKDAY([Date],2) >= 6, 1, 0),
    "Annee_Mois",   FORMAT([Date], "YYYY-MM")
)
```

**Puis marquer comme table de dates** : clic droit `Calendrier` → *Marquer comme table de dates* → colonne `Date`.

---
## 5. Table _Mesures (placeholder)

Modélisation → **Nouvelle table** → coller :

```dax
_Mesures = {BLANK()}
```

Puis **masquer la colonne `Value`** dans le panneau Champs (clic droit → Masquer). La table sert uniquement à héberger les mesures dans des dossiers d'affichage.

---
## 6. Design system — E-commerce

### Palette DataProjectLab E-commerce

| Usage | Couleur | Hex |
|---|---|---|
| Fond principal | Navy foncé | `#1A1F2E` |
| Fond secondaire (cartes) | Navy moyen | `#232836` |
| Accent principal | **Bleu** | `#3B82F6` |
| Performance positive | **Vert teal** | `#10B981` |
| Accent secondaire | **Violet** | `#8B5CF6` |
| Attention | **Orange** | `#F59E0B` |
| Alerte | **Rouge** | `#EF4444` |
| Indigo (variations) | Indigo | `#6366F1` |
| Texte principal | Blanc | `#FFFFFF` |
| Texte secondaire | Gris clair | `#E8E8E8` |
| Texte discret | Gris moyen | `#888888` |

### Typographie

| Usage | Police | Taille |
|---|---|---|
| Titres pages | **Segoe UI** bold | 22-26px |
| Sous-titres (janv-déc 2024) | Segoe UI regular | 13-14px |
| Valeurs KPI | **Segoe UI** bold | 34-42px |
| Labels KPI | Segoe UI regular | 12-13px |
| Variations (pastilles) | Segoe UI | 11px |

### Principes

- Fond de page uniforme `#1A1F2E`
- Cartes de KPI avec fond `#232836`, bordures fines, valeur colorée selon le KPI
- **Chaque KPI a sa couleur dédiée** : CA=bleu, Marge=vert, Commandes=teal, Clients=violet, Panier=orange, Note=rouge
- Pastilles de variation vs N-1 en couleur verte (positif) / rouge (négatif)
- Flèches ↑ ↓ avant le %

---
## 7. Architecture de navigation

### Panneau de navigation latéral

Bande verticale à gauche (~200px de large), fond `#1A1F2E`, présente sur les **5 pages**.

**Contenu en haut** : logo DataProjectLab (carré bleu 40×40) + texte :
- `DataProjectLab` en blanc bold 14px
- `E-commerce` en gris `#888` 12px

**Menu de navigation** (5 items avec icônes) :

| Icône | Label | Page cible |
|---|---|---|
| 📊 | Overview | Page 1 |
| 🛒 | Produit | Page 2 |
| 👥 | Clients | Page 3 |
| 🔽 | Funnel digital | Page 4 |
| ⭐ | Satisfaction | Page 5 |

**Item actif** : fond navy clair `#232836`, texte blanc bold, icône bleue `#3B82F6`, bordure gauche 3px bleue.

**Implémentation** : pour chaque item, Insertion → **Bouton** → Action → **Navigation de page**.

---
## 8. Slicers globaux

**Sur les 5 pages**, bandeau supérieur droit avec 2 groupes de slicers boutons :

### Groupe 1 — Année

| Slicer | Champ | Style | Valeurs |
|---|---|---|---|
| **Année** | `Calendrier[Annee]` | **Boutons arrondis** (single select) | `2022` · `2023` · `2024` |

### Groupe 2 — Trimestre

| Slicer | Champ | Style | Valeurs |
|---|---|---|---|
| **Trimestre** | `Calendrier[Trimestre]` | **Boutons arrondis** (multi-select) | `T1` · `T2` · `T3` · `T4` |

### Style des boutons

- Forme arrondie (border-radius 20px)
- Fond transparent, bordure bleue `#3B82F6` 1px
- Bouton actif : fond bleu `#3B82F6`, texte blanc
- Bouton inactif : texte bleu `#3B82F6`, fond transparent

### Synchronisation

Clic droit sur chaque slicer → *Synchroniser les segments* → cocher les 5 pages (visible ET filtre).

---
## 9. Page 1 — Overview

**Titre** : `Overview`
**Sous-titre** : `Performance globale · janv — déc 2024`

### Ligne 1 : 6 KPI cards

Chaque carte : fond `#232836`, bordure supérieure colorée 3px, valeur en Segoe UI bold 38px colorée, pastille variation en-dessous.

| # | Label | Valeur principale | Pastille variation | Couleur |
|---|---|---|---|---|
| 1 | Chiffre d'affaires | `[CA Total]` → **3,9M** | ↓ -54,3% vs 2023 | 🔵 Bleu `#3B82F6` |
| 2 | Marge totale | `[Marge Totale]` → **1,8M** (46,1% taux) | ↓ -54,2% | 🟢 Vert `#10B981` |
| 3 | Commandes | `[Nb Commandes]` → **3 018** | ↓ -52,8% vs 2023 | 🟢 Teal |
| 4 | Clients actifs | `[Nb Clients]` → **3 000** | ↑ +0,0% | 🟣 Violet `#8B5CF6` |
| 5 | Panier moyen | `[Panier Moyen]` → **1 297** | ↓ -3,1% | 🟠 Orange `#F59E0B` |
| 6 | Note moyenne | `[Note Moyenne]` → **3,8** ★★★★☆ | ↓ -0,1 étoiles | 🔴 Rouge `#EF4444` |

> 💡 Les pastilles utilisent `[Variation CA %]`, `[Variation Marge %]`, etc. (mesures avec UNICHAR flèche).

### Ligne 2 : Evolution CA + Donut catégories

**Gauche — Evolution CA | 2024** (zone courbe)
- Visuel : **Graphique en aires**
- Axe X : `Calendrier[Mois_Nom]`
- Axe Y : `[CA Total]`
- Couleur : bleu `#3B82F6` avec dégradé vers transparent

**Droite — CA par catégorie | Répartition 2024** (donut)
- Visuel : **Graphique en anneau**
- Légende : `dim_products[categorie]`
- Valeurs : `[CA Total]`
- Palette : Ordinateurs bleu, Smartphones teal, Autres gris, Tablettes violet, Audio orange
- Résultat : Ordinateurs 28% · Smartphones 27% · Autres 22% · Tablettes 14% · Audio 8%

### Ligne 3 : CA par canal + Moyens de paiement

**Gauche — CA par canal de vente** (bar chart horizontal)
- Visuel : **Histogramme à barres**
- Axe Y : `fact_orders[canal]`
- Axe X : `[CA Total]`
- Couleurs distinctes par canal : Organic bleu, Social Media teal, Paid Search violet, Direct orange, Email rouge, Affiliate vert
- Format labels : `€ 1023 K` (milliers)

**Droite — Moyen de paiement** (donut)
- Visuel : **Graphique en anneau**
- Légende : `fact_orders[moyen_paiement]`
- Résultat : Carte bancaire 35% · Mobile Money 28% · PayPal 18% · Virement 11% · Cash on Delivery 9%

---
## 10. Page 2 — Produit

**Titre** : `Poduits` (ou `Produits` corrigé)
**Sous-titre** : `Performance & Rentabilité janv — déc 2024`

### Ligne 1 : 4 KPI cards

| # | Label | Mesure | Valeur attendue | Pastille | Couleur |
|---|---|---|---|---|---|
| 1 | Qte vendue | `[Quantité Vendue]` | 12 443 | ↓ -53,2% | 🔵 Bleu |
| 2 | Revenu produits | `[CA Produits]` | € 4M | ↓ -54,3% | 🔵 Bleu |
| 3 | Marge Produits | `[Marge Produits]` | € 2M | ↓ -54,2% | 🟢 Vert |
| 4 | Produits actifs | `[Nb Produits Actifs]` | 30 | ↑ +0,0% | 🟢 Vert |

### Ligne 2 : Top 10 produits + Scatter Revenu/Marge

**Gauche — Top 10 produits — CA** (bar chart horizontal)
- Visuel : **Histogramme à barres** trié DESC
- Axe Y : `dim_products[nom_produit]` (Top 10)
- Axe X : `[CA Total]`
- Format : 616k / 538k / 389k / 314k etc.
- Couleurs dégradées bleu → violet → orange → rouge selon rang
- Top 10 observé : iPhone 15 Pro · MacBook Air M3 · Samsung Galaxy S24 · iPad Air · HP Pavilion 14 · Dell Inspiron 15 · Samsung Tab S9 · GoPro Hero 12 · Sony WH-1000XM5 · Samsung Watch 6

**Droite — Revenu vs Marge par produit** (scatter bubble)
- Visuel : **Nuage de points**
- Axe X : `[CA Produit]` (Revenu)
- Axe Y : `[Marge Produit]`
- Taille bulle : `[Quantité Vendue]`
- Détails : `dim_products[nom_produit]`
- Data labels activés sur les leaders (iPhone 15 Pro, MacBook Air M3)

### Ligne 3 : Détail produits

**Tableau** avec colonnes :

| Colonne | Champ | Format |
|---|---|---|
| Produit | `dim_products[nom_produit]` | Bold |
| Catégorie | `dim_products[categorie]` | Regular |
| Qte | `[Quantité Vendue]` | #,0 |
| Revenu | `[CA Produit]` | € #,0 |
| Marge | `[Marge Produit]` | € #,0 |
| % | `[Taux de Marge Produit]` | ▼ 0,0% (avec flèche rouge) |

Formatage conditionnel : flèche ▼ rouge devant le taux de marge négatif.

---
## 11. Page 3 — Clients

**Titre** : `Clients`
**Sous-titre** : `Segmentation & Valeur janv — déc 2024`

### Ligne 1 : 3 KPI cards

| # | Label | Mesure | Valeur | Pastille | Couleur |
|---|---|---|---|---|---|
| 1 | Clients actifs | `[Nb Clients]` | 3 000 | ↑ +0,0% | 🔵 Bleu |
| 2 | Revenu moy/client | `[CA Moyen par Client]` | € 1 305 | ↓ -54,3% | 🔵 Bleu |
| 3 | Commande moy/client | `[Nb Commandes par Client]` | 1,0 | ↓ -52,8% | 🔵 Bleu |

### Ligne 2 : CA par segment client + Commandes par segment

**Gauche — CA par segment client** (bar chart horizontal)
- Visuel : **Histogramme à barres**
- Axe Y : `dim_customers[segment_client]`
- Axe X : `[CA Total]`
- Palette : Premium violet · Standard bleu · Occasionnel teal · Nouveau orange
- Valeurs : Premium 702K€ · Standard 1273K€ · Occasionnel 1329K€ · Nouveau 612K€

**Milieu — Commandes par segment** (bar chart vertical)
- Visuel : **Histogramme groupé**
- Axe X : `dim_customers[segment_client]`
- Axe Y : `[Nb Commandes]`
- Même palette que la bar chart CA

**Droite — Répartition Clients segment RFM** (donut)
- Visuel : **Graphique en anneau**
- Légende : `clients_rfm_segments[segment_rfm]`
- Valeurs : `COUNT clients`
- Résultat : Dormants 30% · Fidèles 27% · Champions 16% · À réactiver 13% · Nouveaux prometteurs 9% · Gros dépensiers occ. 5%
- Palette RFM dédiée (voir mesure `Couleur Segment RFM`)

### Ligne 3 : CA par segment RFM + Top clients

**Gauche — CA par segment RFM** (bar chart horizontal)
- Visuel : **Histogramme à barres**
- Axe Y : `clients_rfm_segments[segment_rfm]`
- Axe X : `[CA Total]`
- Valeurs : Champions 1468 K€ · Fidèles 1438 K€ · Gros dépensiers occ. 265 K€ · Nouveaux prometteurs 411 K€ · À réactiver 78 K€ · Dormants 256 K€

**Droite — Top clients** (tableau)

| Colonne | Champ |
|---|---|
| ID CLIENT | `dim_customers[customer_id]` |
| SEGMENT | `dim_customers[segment_client]` |
| REVENU | `[CA Total]` (trié DESC) |
| # COMMANDE | `[Nb Commandes]` |

Top observé : CUS01242 Premium 6 460€ 5,0 · CUS02509 Occasionnel 5 691,5€ 5,0 · CUS00043 Premium 4 935€ 5,0 · CUS02307 Occasionnel 4 499€ 3,0

---
## 12. Page 4 — Funnel digital

**Titre** : `Funnel digital`
**Sous-titre** : `Analyse du tunnel d'achat janv — déc 2024`

### Ligne 1 : 4 KPI cards

| # | Label | Mesure | Valeur | Pastille | Couleur |
|---|---|---|---|---|---|
| 1 | Vues totales | `[Nb Vues]` | 1,1k | ↓ -52,0% | 🔵 Bleu |
| 2 | Ajouts panier | `[Nb Ajouts Panier]` | 320 | ↓ -46,7% | 🟢 Vert |
| 3 | Achats | `[Nb Achats]` | 100 | ↓ -41,2% | 🟢 Vert |
| 4 | Taux de conv. | `[Taux Conversion]` | 9,4% | ↑ +1,7pp | 🟠 Orange |

### Ligne 2 : Funnel d'achat + Conversions

**Gauche — Funnel d'achat (View → Cart → Purchase)** (custom funnel)
- Visuel : **Entonnoir** (natif) ou zones empilées personnalisées
- Étapes et valeurs :
  - **Vues** : 1 059 (départ) — bleu foncé
  - **Ajout panier** : 320 (↓ 30,2% des vues) — vert
  - **Checkout** : 149 (↓ 46,6% du panier) — bleu clair
  - **Achat confirmé** : 100 (↓ 67,1% du checkout) — jaune
- Encadré alerte en bas : ⚠ **Point de friction** — *"Perte majeure entre Vues et Panier — 69,8% d'abandons"*

**Droite — Conversion par source** (bar chart horizontal)
- Visuel : **Histogramme à barres**
- Axe Y : `fact_web_logs[source]`
- Axe X : `[Taux Conversion par source]`
- Valeurs : Direct 10,2% · Email 8,6% · Facebook 11,0% · Instagram 8,9% · Google 8,3% · TikTok 10,3%

**Droite (sous source) — Conversion par device** (bar chart horizontal)
- Visuel : **Histogramme à barres**
- Axe Y : `fact_web_logs[device]`
- Axe X : `[Taux Conversion par device]`
- Valeurs : Desktop 8,3% · Mobile 10,7% · Tablette 6,2%

### Ligne 3 : Tableau détaillé

| Colonne | Champ |
|---|---|
| SOURCE | `fact_web_logs[source]` |
| DEVICE | `fact_web_logs[device]` |
| VUES | `[Nb Vues]` |
| PANIER | `[Nb Ajouts Panier]` |
| ACHATS | `[Nb Achats]` |
| TAUX CONVERSION | `[Taux Conversion]` |

---
## 13. Page 5 — Satisfaction client

**Titre** : `Satisfaction client`
**Sous-titre** : `Avis & notes produits janv — déc 2024`

### Ligne 1 : 4 KPI cards

| # | Label | Mesure | Valeur | Pastille | Couleur |
|---|---|---|---|---|---|
| 1 | Note moyenne | `[Note Moyenne]` | 3,8 ★★★★☆ | ↓ -54% | 🔵 Bleu |
| 2 | Nombre d'avis | `[Nb Avis]` | 538 | ↓ -54% | 🔵 Bleu |
| 3 | Produits note <3 | `[Nb Produits Note <3]` | 0 | ↑ +0 | 🔴 Rouge |
| 4 | % Avis positifs | `[Pct Avis Positifs]` | 69% | ↓ -4pp | 🟢 Vert |

### Ligne 2 : Distribution des notes + Produits faibles

**Gauche — Distribution des notes** (bar chart horizontal)
- Visuel : **Histogramme à barres**
- Axe Y : `fact_reviews[rating]` (5, 4, 3, 2, 1)
- Axe X : `[Nb Avis]`
- Couleur conditionnelle : orange pour 3-5 étoiles, rouge pour 1-2 étoiles
- Valeurs : 5★ = 201 · 4★ = 169 · 3★ = 82 · 2★ = 56 · 1★ = 30

**Gauche (dessous) — Evolution mensuelle de la note** (courbe)
- Visuel : **Graphique en courbes**
- Axe X : `Calendrier[Mois_Nom]`
- Axe Y : `[Note Moyenne]`
- Couleur : orange `#F59E0B`

**Droite — Produits avec notes les plus faibles** (bar chart horizontal)
- Visuel : **Histogramme à barres**
- Axe Y : `dim_products[nom_produit]` (bottom 4)
- Axe X : `[Note Moyenne Produit]`
- Valeurs : Dell Inspiron 15 3,4★ · Samsung Tab S9 3,5★ · iPad Air 3,5★ · Sony WH-1000XM5 3,5★

### Ligne 3 : Tableau détail satisfaction

| Colonne | Champ | Format |
|---|---|---|
| PRODUIT | `dim_products[nom_produit]` | Bold |
| CATÉGORIE | `dim_products[categorie]` | Regular |
| NOTE MOY. | `[Note Moyenne Produit]` | ★ pastille colorée (vert si ≥4, orange si 3-4, rouge <3) |
| NB AVIS | `[Nb Avis]` | #,0 |
| TENDANCE | `[Tendance Note]` | → stable / ↑ +0,3 (vert) / ↓ (rouge) |

Exemples : Coque iPhone — Accessoires — ★4,1 — 25 — → stable ; Parfum Hugo Boss — Beaute — ★4,2 — 24 — ↑ +0,3

---
## 14. Mesures DAX — Table `_Mesures`

Toutes les mesures sont organisées par **dossier d'affichage** pour faciliter la maintenance.

### 📂 1. KPIs de base (9 mesures)

```dax
CA Total = 
CALCULATE(
    SUM(fact_order_items[line_revenue]),
    fact_orders[order_status] = "Livree"
)

Marge Totale = 
CALCULATE(
    SUM(fact_order_items[line_margin]),
    fact_orders[order_status] = "Livree"
)

Taux de Marge = 
DIVIDE([Marge Totale], [CA Total], 0)

Nb Commandes = 
CALCULATE(
    DISTINCTCOUNT(fact_orders[order_id]),
    fact_orders[order_status] = "Livree"
)

Nb Clients = DISTINCTCOUNT(dim_customers[customer_id])

Panier Moyen = DIVIDE([CA Total], [Nb Commandes], 0)

Quantité Vendue = 
CALCULATE(
    SUM(fact_order_items[quantite]),
    fact_orders[order_status] = "Livree"
)

Note Moyenne = AVERAGE(fact_reviews[rating])

Nb Avis = COUNTROWS(fact_reviews)
```

### 📂 2. KPIs avancés (6 mesures)

```dax
Nb Produits Actifs = 
CALCULATE(
    DISTINCTCOUNT(dim_products[product_id]),
    fact_order_items[quantite] > 0
)

CA Moyen par Client = DIVIDE([CA Total], [Nb Clients], 0)

Nb Commandes par Client = DIVIDE([Nb Commandes], [Nb Clients], 0)

Nb Produits Note <3 = 
CALCULATE(
    DISTINCTCOUNT(dim_products[product_id]),
    FILTER(
        VALUES(dim_products[product_id]),
        CALCULATE(AVERAGE(fact_reviews[rating])) < 3
    )
)

Pct Avis Positifs = 
DIVIDE(
    CALCULATE(COUNTROWS(fact_reviews), fact_reviews[rating] >= 4),
    [Nb Avis],
    0
)

Tendance Note = 
VAR _note_actuelle = [Note Moyenne]
VAR _note_precedente = CALCULATE([Note Moyenne], DATEADD(Calendrier[Date], -1, MONTH))
VAR _delta = _note_actuelle - _note_precedente
RETURN
SWITCH(
    TRUE(),
    _delta > 0.1, "↑ +" & FORMAT(_delta, "0.0"),
    _delta < -0.1, "↓ " & FORMAT(_delta, "0.0"),
    "→ stable"
)
```

### 📂 3. Variations vs N-1 (6 mesures avec flèches)

```dax
Variation CA % = 
VAR _ca_actuel = [CA Total]
VAR _ca_precedent = CALCULATE([CA Total], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct = DIVIDE(_ca_actuel - _ca_precedent, _ca_precedent, 0)
VAR _format = FORMAT(_pct, "0.0%;0.0%")
RETURN 
IF(_pct > 0, UNICHAR(9650) & " " & _format, UNICHAR(9660) & " " & _format)

Variation Marge % = 
VAR _actuel = [Marge Totale]
VAR _precedent = CALCULATE([Marge Totale], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct = DIVIDE(_actuel - _precedent, _precedent, 0)
VAR _format = FORMAT(_pct, "0.0%;0.0%")
RETURN IF(_pct > 0, UNICHAR(9650) & " " & _format, UNICHAR(9660) & " " & _format)

Variation Commandes % = 
VAR _actuel = [Nb Commandes]
VAR _precedent = CALCULATE([Nb Commandes], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct = DIVIDE(_actuel - _precedent, _precedent, 0)
VAR _format = FORMAT(_pct, "0.0%;0.0%")
RETURN IF(_pct > 0, UNICHAR(9650) & " " & _format, UNICHAR(9660) & " " & _format)

Variation Clients % = 
VAR _actuel = [Nb Clients]
VAR _precedent = CALCULATE([Nb Clients], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct = DIVIDE(_actuel - _precedent, _precedent, 0)
VAR _format = FORMAT(_pct, "0.0%;0.0%")
RETURN IF(_pct > 0, UNICHAR(9650) & " " & _format, UNICHAR(9660) & " " & _format)

Variation Panier % = 
VAR _actuel = [Panier Moyen]
VAR _precedent = CALCULATE([Panier Moyen], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct = DIVIDE(_actuel - _precedent, _precedent, 0)
VAR _format = FORMAT(_pct, "0.0%;0.0%")
RETURN IF(_pct > 0, UNICHAR(9650) & " " & _format, UNICHAR(9660) & " " & _format)

Variation Note % = 
VAR _actuel = [Note Moyenne]
VAR _precedent = CALCULATE([Note Moyenne], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _delta = _actuel - _precedent
RETURN 
IF(_delta > 0, UNICHAR(9650) & " +" & FORMAT(_delta, "0.0"),
   UNICHAR(9660) & " " & FORMAT(_delta, "0.0"))
```

---
## 15. Mesures DAX — Couleurs, Funnel & Web

### 📂 4. Couleurs Variation (formatage conditionnel des pastilles)

Utilisées dans **Format → Couleur d'arrière-plan → Par formule** sur chaque carte KPI.

```dax
Couleur Variation CA = 
VAR _pct = DIVIDE([CA Total] - CALCULATE([CA Total], SAMEPERIODLASTYEAR(Calendrier[Date])), 
                   CALCULATE([CA Total], SAMEPERIODLASTYEAR(Calendrier[Date])), 0)
RETURN SWITCH(TRUE(), _pct > 0, "#10B981", _pct < 0, "#EF4444", "#888888")

Couleur Variation Marge = 
VAR _pct = DIVIDE([Marge Totale] - CALCULATE([Marge Totale], SAMEPERIODLASTYEAR(Calendrier[Date])),
                   CALCULATE([Marge Totale], SAMEPERIODLASTYEAR(Calendrier[Date])), 0)
RETURN SWITCH(TRUE(), _pct > 0, "#10B981", _pct < 0, "#EF4444", "#888888")

-- Répéter le même pattern pour Commandes, Clients, Panier, Note
```

### 📂 5. Couleurs Segment RFM (Page 3)

```dax
Couleur Segment RFM = 
SWITCH(
    SELECTEDVALUE(clients_rfm_segments[segment_rfm]),
    "Champions",                    "#10B981",
    "Fidèles",                      "#8B5CF6",
    "Gros dépensiers occasionnels", "#F59E0B",
    "Nouveaux prometteurs",         "#3B82F6",
    "À réactiver",                  "#EF4444",
    "Dormants",                     "#888888"
)
```

### 📂 6. Funnel digital (Page 4)

```dax
Nb Sessions = DISTINCTCOUNT(fact_web_logs[session_id])

Nb Vues = 
CALCULATE(
    COUNTROWS(fact_web_logs),
    SEARCH("fiche_produit", fact_web_logs[page], 1, 0) > 0
)

Nb Ajouts Panier = 
CALCULATE(
    DISTINCTCOUNT(fact_web_logs[session_id]),
    SEARCH("panier", fact_web_logs[page], 1, 0) > 0
)

Nb Checkout = 
CALCULATE(
    DISTINCTCOUNT(fact_web_logs[session_id]),
    SEARCH("checkout", fact_web_logs[page], 1, 0) > 0
)

Nb Achats = 
CALCULATE(
    DISTINCTCOUNT(fact_web_logs[session_id]),
    SEARCH("confirmation", fact_web_logs[page], 1, 0) > 0
)

Taux Conversion = DIVIDE([Nb Achats], [Nb Vues], 0)

-- Transitions du funnel
Taux Panier/Vues = DIVIDE([Nb Ajouts Panier], [Nb Vues], 0)
Taux Checkout/Panier = DIVIDE([Nb Checkout], [Nb Ajouts Panier], 0)
Taux Achat/Checkout = DIVIDE([Nb Achats], [Nb Checkout], 0)

-- Point de friction dynamique
Abandon Vues->Panier = 
VAR _abandon = 1 - [Taux Panier/Vues]
RETURN "Perte majeure entre Vues et Panier — " & FORMAT(_abandon, "0.0%") & " d'abandons"
```

### 📂 7. Segment Produits (Page 2)

```dax
CA Produit = SUMX(fact_order_items, fact_order_items[line_revenue])

Marge Produit = SUMX(fact_order_items, fact_order_items[line_margin])

Taux de Marge Produit = DIVIDE([Marge Produit], [CA Produit], 0)

Note Moyenne Produit = 
CALCULATE(
    AVERAGE(fact_reviews[rating]),
    ALLEXCEPT(dim_products, dim_products[product_id])
)
```

---
## 16. Checklist de validation

### Import & modèle
- [ ] 7 fichiers importés sans erreur
- [ ] Auto Date/Time désactivé
- [ ] Table `Calendrier` créée et marquée comme table de dates
- [ ] Table `_Mesures` créée (colonne Value masquée)
- [ ] 10 relations actives, aucun chemin ambigu
- [ ] Aucune relation M:M bidirectionnelle

### Mesures
- [ ] 9 mesures KPIs de base
- [ ] 6 mesures KPIs avancés
- [ ] 6 mesures Variations vs N-1 avec flèches ↑ ↓
- [ ] 6 mesures Couleur Variation (formatage conditionnel)
- [ ] 1 mesure Couleur Segment RFM
- [ ] 8 mesures Funnel digital
- [ ] 4 mesures Segment Produits

### Valeurs attendues sans filtre (année 2024)

| Mesure | Valeur |
|---|---|
| `[CA Total]` | 3,9 M € |
| `[Marge Totale]` | 1,8 M € |
| `[Taux de Marge]` | 46,1% |
| `[Nb Commandes]` | 3 018 |
| `[Nb Clients]` | 3 000 |
| `[Panier Moyen]` | 1 297 € |
| `[Note Moyenne]` | 3,8 |
| `[Quantité Vendue]` | 12 443 |
| `[Nb Produits Actifs]` | 30 |
| `[Nb Vues]` | 1 059 |
| `[Nb Ajouts Panier]` | 320 |
| `[Nb Achats]` | 100 |
| `[Taux Conversion]` | 9,4% |
| `[Nb Avis]` | 538 |
| `[Pct Avis Positifs]` | 69% |

### Navigation & slicers
- [ ] Panneau latéral DataProjectLab sur les 5 pages
- [ ] Icône + label actif visible selon la page courante
- [ ] Slicer Année (boutons 2022/2023/2024) en haut à droite
- [ ] Slicer Trimestre (boutons T1/T2/T3/T4) en haut à droite
- [ ] Slicers synchronisés sur les 5 pages

### Pages
- [ ] Page 1 Overview : 6 KPI + courbe CA + donut catégories + bar canal + donut paiement
- [ ] Page 2 Produit : 4 KPI + Top 10 produits + scatter Revenu/Marge + tableau détail
- [ ] Page 3 Clients : 3 KPI + CA segment client + Commandes segment + donut RFM + CA segment RFM + Top clients
- [ ] Page 4 Funnel : 4 KPI + funnel d'achat + Conversion source + Conversion device + tableau
- [ ] Page 5 Satisfaction : 4 KPI + Distribution notes + Evolution mensuelle + Produits faibles + tableau détail

---
## 17. Storytelling — Ordre de présentation

### Séquence narrative pour le comité de direction

1. **Performance globale** (page Overview)  
   → *"Voici où on en est : CA 3,9M (-54% vs 2023), marge 1,8M, 3 018 commandes"*

2. **Moteurs de performance** (page Produit)  
   → *"iPhone 15 Pro et MacBook Air M3 concentrent 25% du CA. 30 produits actifs portent toute la performance."*

3. **Valeur client** (page Clients)  
   → *"3 000 clients actifs, revenu moyen € 1 305 (-54%). Segments RFM : 43% dormants + à réactiver — priorité reactivation"*

4. **Conversion digitale** (page Funnel)  
   → *"Taux conversion 9,4% (+1,7pp vs 2023). Point de friction majeur : 69,8% d'abandons entre Vues et Panier"*

5. **Qualité perçue** (page Satisfaction)  
   → *"Note 3,8/5, 69% avis positifs. 4 produits sous la barre des 3,5 étoiles à surveiller (Dell Inspiron 15, Samsung Tab S9, iPad Air, Sony WH-1000XM5)"*

### Message final

> L'apprenant ne doit pas seulement savoir créer un dashboard.  
> Il doit être capable de répondre à la question :
>
> **"Que doit faire ShopAfrica+ dans les 3 prochains mois ?"**

### Axes d'action suggérés (dérivés du dashboard)

1. **Stopper l'hémorragie CA** (-54% vs 2023) — investigation urgente sur ordinateurs + smartphones
2. **Reconquête clients dormants** (30% de la base) — campagne email personnalisée
3. **Fix du tunnel d'achat** — point de friction Vues→Panier (69,8% d'abandons)
4. **Stabiliser la qualité produit** — 4 produits à surveiller avec notes < 3,5 étoiles

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.